# PoC 4: LLM-Powered Summaries + Chat-over-DB

**Question:** Are Claude-powered summaries and natural-language queries actually useful, or gimmicky?

This notebook explores three capabilities:
1. **Daily summary generation** — Condense 24h of QC metrics into a readable paragraph
2. **Chat-over-DB** — Convert natural language questions to SQL and execute them
3. **Anomaly explanation** — Ask Claude to hypothesize causes of deviations from baseline

All database access is read-only via `_common.load_db`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
import json
import sqlite3
from datetime import datetime, timedelta

import pandas as pd
from anthropic import Anthropic

from _common import load_db, db_schema_text

DB_PATH = os.path.join('..', 'data', 'monitor.db')

# Initialize client — requires ANTHROPIC_API_KEY env var
client = Anthropic()
MODEL = "claude-sonnet-4-6"  # Use Sonnet for cost efficiency in PoC

# Running token counters for cost estimation
total_input_tokens = 0
total_output_tokens = 0
total_cache_creation_tokens = 0
total_cache_read_tokens = 0

print(f"Model: {MODEL}")
print(f"DB path: {os.path.abspath(DB_PATH)}")
print(f"Schema length: {len(db_schema_text())} chars")

## 1. Daily summary generation

In [ ]:
def gather_daily_data(db_path: str, lookback_hours: int = 24) -> dict:
    """Query last N hours of monitoring data for summary generation."""
    assert os.path.isfile(db_path), f"Database not found: {db_path}"
    assert lookback_hours > 0, "lookback_hours must be positive"

    conn = load_db(db_path)
    cutoff = (datetime.now() - timedelta(hours=lookback_hours)).isoformat()

    try:
        # Processed files
        files_df = pd.read_sql_query(
            "SELECT id, file_path, status, session_name, chunk_datetime, "
            "duration_sec, num_channels "
            "FROM processed_files "
            "WHERE processed_at >= ? ORDER BY chunk_datetime",
            conn, params=[cutoff]
        )

        file_ids = files_df['id'].tolist()
        if len(file_ids) == 0:
            # Fall back to most recent files if none in last 24h
            files_df = pd.read_sql_query(
                "SELECT id, file_path, status, session_name, chunk_datetime, "
                "duration_sec, num_channels "
                "FROM processed_files ORDER BY processed_at DESC LIMIT 20",
                conn
            )
            file_ids = files_df['id'].tolist()

        placeholders = ','.join('?' for _ in file_ids)

        # Alerts
        alerts_df = pd.read_sql_query(
            f"SELECT alert_type, severity, message, sent_at "
            f"FROM alerts WHERE file_id IN ({placeholders}) "
            f"ORDER BY sent_at DESC LIMIT 50",
            conn, params=file_ids
        )

        # Seizure events
        seizure_df = pd.read_sql_query(
            f"SELECT file_id, channel, onset_sec, duration_sec, "
            f"spike_count, peak_amplitude, severity "
            f"FROM seizure_events WHERE file_id IN ({placeholders}) "
            f"ORDER BY severity DESC LIMIT 50",
            conn, params=file_ids
        )

        # Evoked summary
        evoked_df = pd.read_sql_query(
            f"SELECT file_id, num_stimuli_detected, num_traces_extracted, "
            f"mean_peak_amplitude "
            f"FROM evoked_summary WHERE file_id IN ({placeholders})",
            conn, params=file_ids
        )

        # Stim QC
        stim_df = pd.read_sql_query(
            f"SELECT file_id, stim_channel, charge_nC, frequency_hz, "
            f"total_pulses, delivery_pct "
            f"FROM stim_qc WHERE file_id IN ({placeholders})",
            conn, params=file_ids
        )
    finally:
        conn.close()

    daily_data = {
        "time_window": f"Last {lookback_hours} hours (cutoff: {cutoff})",
        "files": {
            "total": len(files_df),
            "by_status": files_df['status'].value_counts().to_dict() if len(files_df) > 0 else {},
            "sessions": files_df['session_name'].nunique() if len(files_df) > 0 else 0,
            "total_duration_min": round(files_df['duration_sec'].sum() / 60, 1) if len(files_df) > 0 else 0,
        },
        "alerts": {
            "total": len(alerts_df),
            "by_severity": alerts_df['severity'].value_counts().to_dict() if len(alerts_df) > 0 else {},
            "by_type": alerts_df['alert_type'].value_counts().to_dict() if len(alerts_df) > 0 else {},
            "recent": alerts_df.head(5).to_dict(orient='records') if len(alerts_df) > 0 else [],
        },
        "seizures": {
            "total_events": len(seizure_df),
            "max_severity": seizure_df['severity'].max() if len(seizure_df) > 0 else None,
            "events": seizure_df.head(10).to_dict(orient='records') if len(seizure_df) > 0 else [],
        },
        "evoked_responses": {
            "files_with_evoked": len(evoked_df),
            "mean_stimuli_detected": round(evoked_df['num_stimuli_detected'].mean(), 1) if len(evoked_df) > 0 else None,
            "mean_peak_amplitude": round(evoked_df['mean_peak_amplitude'].mean(), 3) if len(evoked_df) > 0 else None,
        },
        "stimulation": {
            "files_with_stim": len(stim_df),
            "mean_delivery_pct": round(stim_df['delivery_pct'].mean(), 1) if len(stim_df) > 0 else None,
            "min_delivery_pct": round(stim_df['delivery_pct'].min(), 1) if len(stim_df) > 0 else None,
            "charge_range_nC": [round(stim_df['charge_nC'].min(), 1), round(stim_df['charge_nC'].max(), 1)] if len(stim_df) > 0 else None,
        },
    }
    return daily_data


daily_data = gather_daily_data(DB_PATH)
print(json.dumps(daily_data, indent=2, default=str))

In [ ]:
SYSTEM_PROMPT = f"""You are a neurophysiology QC assistant for a brain stimulation research lab.
You analyze monitoring data from KMrecorder systems that record neural signals
and deliver electrical stimulation to rodent brains.

Database schema:
{db_schema_text()}

Your job is to produce concise daily summaries highlighting:
1. How many files were processed and their status
2. Any alerts or anomalies detected
3. Seizure events if any
4. Stimulation delivery quality
5. Signal quality trends
6. Actionable recommendations

Keep summaries under 200 words. Use clinical/technical language appropriate
for a neuroscience research team."""


def generate_daily_summary(daily_data: dict) -> str:
    """Generate a daily QC summary via Claude API with prompt caching."""
    assert isinstance(daily_data, dict), "daily_data must be a dict"
    assert len(daily_data) > 0, "daily_data must not be empty"

    global total_input_tokens, total_output_tokens
    global total_cache_creation_tokens, total_cache_read_tokens

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            system=[{
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"}
            }],
            messages=[{
                "role": "user",
                "content": (
                    "Generate a daily QC summary for the following data:\n\n"
                    f"{json.dumps(daily_data, indent=2, default=str)}"
                )
            }]
        )
    except Exception as exc:
        return f"[API ERROR] {type(exc).__name__}: {exc}"

    # Track token usage
    usage = response.usage
    total_input_tokens += usage.input_tokens
    total_output_tokens += usage.output_tokens
    if hasattr(usage, 'cache_creation_input_tokens'):
        total_cache_creation_tokens += usage.cache_creation_input_tokens or 0
    if hasattr(usage, 'cache_read_input_tokens'):
        total_cache_read_tokens += usage.cache_read_input_tokens or 0

    return response.content[0].text


summary = generate_daily_summary(daily_data)
print("=" * 60)
print("DAILY QC SUMMARY")
print("=" * 60)
print(summary)
print("=" * 60)
print(f"\nToken usage so far:")
print(f"  Input:          {total_input_tokens}")
print(f"  Output:         {total_output_tokens}")
print(f"  Cache created:  {total_cache_creation_tokens}")
print(f"  Cache read:     {total_cache_read_tokens}")

## 2. Chat-over-DB: Natural language to SQL

In [ ]:
SQL_SYSTEM = f"""You are a SQL query generator for a neurophysiology monitoring database.

Schema:
{db_schema_text()}

Rules:
- Generate ONLY SELECT queries — never INSERT, UPDATE, DELETE, DROP, or ALTER
- Always include LIMIT to prevent huge result sets (default LIMIT 100)
- Return ONLY the SQL query, no explanation
- Use appropriate JOINs when the question spans multiple tables
- For time-based questions, chunk_datetime is in ISO format
"""


def nl_to_sql(question: str) -> str:
    """Convert a natural language question to SQL."""
    assert isinstance(question, str), "question must be a string"
    assert len(question.strip()) > 0, "question must not be empty"

    global total_input_tokens, total_output_tokens
    global total_cache_creation_tokens, total_cache_read_tokens

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=300,
            system=[{
                "type": "text",
                "text": SQL_SYSTEM,
                "cache_control": {"type": "ephemeral"}
            }],
            messages=[{"role": "user", "content": question}]
        )
    except Exception as exc:
        return f"-- API ERROR: {type(exc).__name__}: {exc}"

    usage = response.usage
    total_input_tokens += usage.input_tokens
    total_output_tokens += usage.output_tokens
    if hasattr(usage, 'cache_creation_input_tokens'):
        total_cache_creation_tokens += usage.cache_creation_input_tokens or 0
    if hasattr(usage, 'cache_read_input_tokens'):
        total_cache_read_tokens += usage.cache_read_input_tokens or 0

    sql = response.content[0].text.strip()
    # Strip markdown code fences if present
    if sql.startswith("```"):
        sql = sql.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    return sql


def execute_safe_query(sql: str, db_path: str = DB_PATH) -> pd.DataFrame:
    """Execute a SQL query in read-only mode. Rejects non-SELECT."""
    assert isinstance(sql, str), "sql must be a string"
    normalized = sql.strip().upper()
    assert normalized.startswith("SELECT"), (
        f"Only SELECT queries allowed, got: {sql[:50]}"
    )

    conn = load_db(db_path)
    try:
        df = pd.read_sql_query(sql, conn)
    finally:
        conn.close()
    return df


def ask_db(question: str) -> tuple:
    """Full pipeline: question -> SQL -> execute -> explain.

    Returns (sql, dataframe, explanation) on success,
    or (sql, None, error_message) on failure.
    """
    assert isinstance(question, str), "question must be a string"
    assert len(question.strip()) > 0, "question must not be empty"

    global total_input_tokens, total_output_tokens

    sql = nl_to_sql(question)

    # Attempt execution
    try:
        df = execute_safe_query(sql)
    except Exception as exc:
        return sql, None, f"[EXECUTION ERROR] {type(exc).__name__}: {exc}"

    # Generate explanation
    try:
        explain_response = client.messages.create(
            model=MODEL,
            max_tokens=200,
            messages=[{
                "role": "user",
                "content": (
                    f"Question: {question}\n"
                    f"SQL: {sql}\n"
                    f"Results ({len(df)} rows):\n"
                    f"{df.head(10).to_string()}\n\n"
                    "Explain the results in one sentence."
                )
            }]
        )
        usage = explain_response.usage
        total_input_tokens += usage.input_tokens
        total_output_tokens += usage.output_tokens
        explanation = explain_response.content[0].text
    except Exception as exc:
        explanation = f"[Explanation unavailable: {exc}]"

    return sql, df, explanation

In [ ]:
TEST_QUESTIONS = [
    "How many files were processed in the last week?",
    "Which sessions have the most seizure events?",
    "What is the average stimulation delivery percentage across all sessions?",
    "Show me files with artifact percentage above 50%",
    "Which channels have the worst signal quality?",
    "How has peak amplitude changed over the last 30 days?",
    "Are there any sessions where stimulation delivery dropped below 90%?",
    "What's the average processing time per file?",
    "Show me all alerts from the last 3 days",
    "Which session has the most files processed?",
]

results = []
successes = 0

for i, question in enumerate(TEST_QUESTIONS):
    print(f"\n{'=' * 60}")
    print(f"Q{i+1}: {question}")
    print("-" * 60)

    sql, df, explanation = ask_db(question)

    print(f"SQL: {sql}")
    print()

    if df is not None:
        successes += 1
        status = "OK"
        print(f"Result: {len(df)} rows")
        if len(df) > 0:
            print(df.head(5).to_string())
        print(f"\nExplanation: {explanation}")
    else:
        status = "FAIL"
        print(f"Error: {explanation}")

    results.append({
        "question": question,
        "status": status,
        "sql": sql,
        "rows": len(df) if df is not None else 0,
    })

sql_success_rate = successes / len(TEST_QUESTIONS)

print(f"\n{'=' * 60}")
print(f"SCORECARD: {successes}/{len(TEST_QUESTIONS)} successful queries")
print(f"Success rate: {sql_success_rate:.0%}")
print("=" * 60)
for r in results:
    print(f"  [{r['status']:4s}] {r['question'][:50]}... ({r['rows']} rows)")

## 3. Anomaly explanation

In [ ]:
def explain_anomaly(session_dir: str, feature_stats: dict, baseline_stats: dict) -> str:
    """Ask Claude to hypothesize why a session deviates from baseline."""
    assert isinstance(session_dir, str), "session_dir must be a string"
    assert isinstance(feature_stats, dict), "feature_stats must be a dict"
    assert isinstance(baseline_stats, dict), "baseline_stats must be a dict"

    global total_input_tokens, total_output_tokens

    prompt = f"""A recording session shows anomalous evoked responses compared to baseline.

Session: {session_dir}

Current session stats (mean across epochs):
{json.dumps(feature_stats, indent=2)}

Historical baseline (mean +/- std):
{json.dumps(baseline_stats, indent=2)}

What are the most likely causes of these deviations? Consider:
- Electrode issues (impedance changes, lead drift, breakage)
- Stimulation problems (charge delivery, contact issues)
- Biological changes (tissue response, inflammation, seizure activity)
- Technical artifacts (noise, grounding, cable movement)

Give your top 3 hypotheses ranked by likelihood, each in one sentence."""

    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}]
        )
    except Exception as exc:
        return f"[API ERROR] {type(exc).__name__}: {exc}"

    usage = response.usage
    total_input_tokens += usage.input_tokens
    total_output_tokens += usage.output_tokens

    return response.content[0].text


# Query an example session's features vs. baseline
conn = load_db(DB_PATH)
try:
    # Find a session with evoked data
    session_row = pd.read_sql_query(
        "SELECT DISTINCT pf.session_dir, pf.session_name "
        "FROM evoked_features ef "
        "JOIN processed_files pf ON pf.id = ef.file_id "
        "LIMIT 1",
        conn
    )

    if len(session_row) > 0:
        target_session = session_row.iloc[0]['session_dir']
        feature_cols = [
            'peak_amplitude', 'trough_amplitude', 'peak_to_trough',
            'rms_amplitude', 'peak_latency_ms', 'line_length',
            'template_correlation', 'recovery_tau'
        ]
        col_list = ', '.join(f'ef.{c}' for c in feature_cols)

        # Current session stats
        session_features = pd.read_sql_query(
            f"SELECT {col_list} FROM evoked_features ef "
            f"JOIN processed_files pf ON pf.id = ef.file_id "
            f"WHERE pf.session_dir = ? AND ef.is_artifact = 0",
            conn, params=[target_session]
        )

        # Global baseline
        baseline_features = pd.read_sql_query(
            f"SELECT {col_list} FROM evoked_features ef "
            f"WHERE ef.is_artifact = 0",
            conn
        )

        feature_stats = session_features[feature_cols].mean().to_dict()
        baseline_stats = {}
        for col in feature_cols:
            mean_val = baseline_features[col].mean()
            std_val = baseline_features[col].std()
            baseline_stats[col] = {
                "mean": round(float(mean_val), 4) if pd.notna(mean_val) else None,
                "std": round(float(std_val), 4) if pd.notna(std_val) else None,
            }
        # Round feature_stats
        feature_stats = {
            k: round(float(v), 4) if pd.notna(v) else None
            for k, v in feature_stats.items()
        }

        print(f"Session: {target_session}")
        print(f"Session epochs: {len(session_features)}")
        print(f"Baseline epochs: {len(baseline_features)}")
        print()

        explanation = explain_anomaly(target_session, feature_stats, baseline_stats)
        print("ANOMALY EXPLANATION:")
        print(explanation)
    else:
        print("No evoked feature data found in database. Skipping anomaly explanation.")
finally:
    conn.close()

## 4. Cost estimate

In [ ]:
# Sonnet pricing (as of 2025):
# Input:  $3.00 / 1M tokens
# Output: $15.00 / 1M tokens
# Cache write: $3.75 / 1M tokens
# Cache read:  $0.30 / 1M tokens

PRICE_INPUT = 3.00 / 1_000_000
PRICE_OUTPUT = 15.00 / 1_000_000
PRICE_CACHE_WRITE = 3.75 / 1_000_000
PRICE_CACHE_READ = 0.30 / 1_000_000

# Actual cost from this run
actual_cost = (
    total_input_tokens * PRICE_INPUT
    + total_output_tokens * PRICE_OUTPUT
    + total_cache_creation_tokens * PRICE_CACHE_WRITE
    + total_cache_read_tokens * PRICE_CACHE_READ
)

print("TOKEN USAGE FROM THIS RUN")
print(f"  Input tokens:          {total_input_tokens:,}")
print(f"  Output tokens:         {total_output_tokens:,}")
print(f"  Cache creation tokens: {total_cache_creation_tokens:,}")
print(f"  Cache read tokens:     {total_cache_read_tokens:,}")
print(f"  Actual cost:           ${actual_cost:.4f}")

# Projected daily usage: 1 summary + 10 queries + 3 anomaly explanations
# This notebook ran: 1 summary + 10 queries (2 API calls each) + 1 anomaly = 22 API calls
# Daily projection: 1 summary + 10 queries (2 each) + 3 anomalies = 24 API calls
# Scale factor vs. this run
num_api_calls_this_run = 1 + 10 * 2 + 1  # summary + queries + anomaly
num_api_calls_daily = 1 + 10 * 2 + 3
scale_factor = num_api_calls_daily / max(num_api_calls_this_run, 1)

daily_cost_no_cache = (
    (total_input_tokens + total_cache_creation_tokens + total_cache_read_tokens)
    * PRICE_INPUT + total_output_tokens * PRICE_OUTPUT
) * scale_factor

daily_cost_with_cache = actual_cost * scale_factor

print(f"\nPROJECTED DAILY COST")
print(f"  Without caching: ${daily_cost_no_cache:.4f}/day (${daily_cost_no_cache * 30:.2f}/month)")
print(f"  With caching:    ${daily_cost_with_cache:.4f}/day (${daily_cost_with_cache * 30:.2f}/month)")
if daily_cost_no_cache > 0:
    savings_pct = (1 - daily_cost_with_cache / daily_cost_no_cache) * 100
    print(f"  Cache savings:   {savings_pct:.1f}%")

## 5. Verdict

In [ ]:
# Criteria:
# (a) Summaries worth reading — subjective, noted here
# (b) Chat-over-DB: correct SQL on >= 8/10 questions

if sql_success_rate >= 0.8:
    verdict = "SHIP"
    reason = (
        f"SQL generation succeeds {sql_success_rate:.0%}. "
        f"Summaries TBD (review 3 sample days)."
    )
elif sql_success_rate >= 0.6:
    verdict = "ITERATE"
    reason = (
        f"SQL at {sql_success_rate:.0%} — needs better schema prompting "
        f"or few-shot examples."
    )
else:
    verdict = "DROP"
    reason = (
        f"SQL only {sql_success_rate:.0%} correct — too unreliable "
        f"for a non-expert user."
    )

print(f"VERDICT: {verdict}")
print(f"Reason:  {reason}")
print(f"\nMonthly cost estimate: ${daily_cost_with_cache * 30:.2f} (with caching)")